# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10 Первое число по индексу, второе по индексу iloc
2. Потому что пересекается только Шымкент
    Алматы      NaN
    Астана      NaN
    Шымкент     3
3. 2 2  Так как 2 строки и 2 столбца
4. Series   DataFrame   Потому что первым мы обращаемся к слобцу как к отдельному объекту, во втором мы выбираем какие поля нужно показать в DataFrame
5. [1,2] 5 Индексы элементов больше 1 и значение на первой строке и втором столбце   

In [2]:
# 1
import pandas as pd


s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [61]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum(), (df["order_date"] >= "2026-10").sum())

сентябрь: 2782 | октябрь: 2681 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [45]:
# 1
print('Сколько строк и столбцов?', df.shape)
print('Какие типы у столбцов?')
df.info()
print("Сколько пропусков в `delivery_days`?", 5463 - 5303)


Сколько строк и столбцов? (5463, 8)
Какие типы у столбцов?
<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 672.1 KB
Сколько пропусков в `delivery_days`? 160


In [20]:
print("Сколько разных значений в `city`, `category`, `status`, `customer_id`?")
print(df["city"].unique())
print(df["category"].unique())
print(df["status"].unique())
print(df["customer_id"].unique())

Сколько разных значений в `city`, `category`, `status`, `customer_id`?
<ArrowStringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str
<ArrowStringArray>
['Одежда', 'Красота', 'Дом', 'Книги', 'Электроника']
Length: 5, dtype: str
<ArrowStringArray>
['оплачен', 'отменён']
Length: 2, dtype: str
<ArrowStringArray>
['C00681', 'C01210', 'C01408', 'C01529', 'C00657', 'C00763', 'C01730',
 'C00896', 'C01349', 'C01934',
 ...
 'C00042', 'C00468', 'C01308', 'C01960', 'C00430', 'C00645', 'C01442',
 'C00018', 'C01650', 'C01057']
Length: 2003, dtype: str


1. Пропуски в столбце со сроком доставки. Чем их заполнять? Удалить строки с пустым значением?
2. По-разному написаны города, нужно привести их к 1 виду чтобы агрегировать и фильтровать без проблем.
3. Столбец amount нужно привести к float, так как это сумма заказа.

Нужно почистить данные по этим столбцам.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [29]:
# 2
df[(df["customer_id"] != "TEST") & (df['status'] == 'оплачен')][["amount"]].describe()

,amount
count,4901.000000
mean,14672.391349
std,13249.340787
min,720.000000
25%,5880.000000
50%,10300.000000
75%,18690.000000
max,139040.000000


Выходит среднее = 14672.39 , медиана(50-персентиль) = 10300, 25-персентиль = 5880 и 75-персентиль = 18690.
Лучше показать медиану, так как в данных есть аномалии(выбросы), а медиана не чуствительна к ним в отличии от средней арифметической.
Если сравнивать с другими квартилями, то все равно побеждает медиана в вопросе 'Какой чек «типичный»?' так как именно она находится по середине.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [33]:
# Взял за основу функцию из второй домашки
def clean_city(s: str) -> str:
    new_s = s.replace(' ', '').capitalize()
    cities = {
        'Almaty': 'Алматы',
        'Astana': 'Астана',
        'Karaganda': 'Караганда',
        'Aqtobe': 'Актобе',
        'Shymkent': 'Шымкент',
        'Atyrau': 'Атырау'
    }
    if new_s in cities.values():
        return new_s
    return cities.get(new_s, 'Не найден')

In [46]:
# 3
df["city"] = df["city"].apply(clean_city)

#df['city'].unique() # проверка

canceled = df[(df['status'] != 'оплачен')]


print(canceled['city'].value_counts())

city
Алматы       172
Шымкент      164
Астана       123
Караганда     48
Актобе        41
Атырау        11
Name: count, dtype: int64


Логично что выделяются 3 больших города, но нужно еще проверить с учетом сколько всего заказов. То есть почитать именно долю.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [84]:
# 4
df_paid = df[(df["customer_id"] != "TEST") & (df['status'] == 'оплачен')]
shymkent_paid_sep = df_paid[(df_paid["city"] == "Шымкент") & (df_paid["order_date"] < "2026-10")]['amount'].sum()
shymkent_paid_oct = df_paid[(df_paid["city"] == "Шымкент") & (df_paid["order_date"] >= "2026-10")]['amount'].sum()


almaty_paid_sep = df_paid[(df_paid["city"] == "Алматы") & (df_paid["order_date"] < "2026-10")]['amount'].sum()
almaty_paid_oct = df_paid[(df_paid["city"] == "Алматы") & (df_paid["order_date"] >= "2026-10")]['amount'].sum()

shymkent_paid_diff = round((shymkent_paid_sep - shymkent_paid_oct)/shymkent_paid_sep * 100, 2)
almaty_paid_diff = round((almaty_paid_sep - almaty_paid_oct)/almaty_paid_sep * 100, 2)

print('Шымкент', shymkent_paid_diff)
print('Алматы', almaty_paid_diff)
print('В шымкенте ситуация хуже в ', round(shymkent_paid_diff / almaty_paid_diff), ' раза в %.')
print('Нужно исследовать данные Шымкента и понять что привело к ухудшению.')


Шымкент 18.3
Алматы 5.67
В шымкенте ситуация хуже в  3  раза в %.
Нужно исследовать данные Шымкента и понять что привело к ухудшению.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [77]:
# 5
cancel_sh = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print('Всего отменненых заказов ', len(cancel_sh['order_id'].unique()))
print('В первоначальном варианте была неправильная фильтрация и подсчет строк с дублями')

Всего отменненых заказов  160
В первоначальном варианте была неправильная фильтрация и подсчет строк с дублями


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [88]:
# 6
df['customer_id'].value_counts()

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
          ..
C00645     1
C01442     1
C00018     1
C01650     1
C01057     1
Name: count, Length: 2003, dtype: int64

In [102]:
df.loc[(df["customer_id"] == 'C01818'), ["order_id", "order_date", "city", "amount", "status"]]

,order_id,order_date,city,amount,status
57,101125,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,100096,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,101860,2026-09-24 18:00,Астана,1490.0,оплачен
2617,101462,2026-09-29 09:00,Астана,22170.0,оплачен
2776,101495,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,103416,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,102644,2026-10-18 08:00,Астана,13600.0,оплачен
4610,104001,2026-10-21 19:00,Алматы,21990.0,оплачен


Есть ли среди его заказов повторяющиеся `order_id`?
Ответ: Да есть полные дубли заказов.

Что это значит для выручки, которую мы считали весь урок?
Ответ: Выручка посчитана неправильно, так как значение по столбцу выручки тоже задвоились.


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [127]:
# 7
Q1 = df_paid["amount"].quantile(0.25)
Q3 = df_paid["amount"].quantile(0.75)
IQR = Q3 - Q1
anomaly_limit = Q3 + (1.5 * IQR)

anomaly_df_paid = df_paid[df_paid['amount'] > anomaly_limit]
anomaly_df_paid_cnt = anomaly_df_paid['amount'].count()
print('Сколько заказов выше границы? Ответ ', anomaly_df_paid_cnt)
print(anomaly_df_paid_cnt / df_paid['order_id'].count() * 100)
print(anomaly_df_paid["category"].value_counts())
print("какая у них категория чаще всего? Ответ Электроника")

Сколько заказов выше границы? Ответ  320
6.529279738828811
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
какая у них категория чаще всего? Ответ Электроника


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [166]:
# 8
# откат очистки столбца с городами для чистоты эксперимента
df_raw = pd.read_csv("orders.csv")

canceled_series = df_raw[(df_raw['status'] == 'отменён')]['city'].value_counts(sort=True, ascending=False)
all_series = df_raw['city'].value_counts(sort=True, ascending=False)

prc_series = canceled_series / all_series * 100
print(prc_series[['шымкент', 'Шымкент', 'Shymkent']].sort_values(ascending=False))

city
Shymkent    23.076923
Шымкент     21.518987
шымкент      3.703704
Name: count, dtype: float64


Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная.

Проверка доли у каждого написания Шымкента.

In [159]:
print(all_series[['шымкент', 'Шымкент', 'Shymkent']])

city
шымкент      54
Шымкент     711
Shymkent     39
Name: count, dtype: int64


Сколько заказов у каждого написания?
Ответ:

шымкент      54
Шымкент     711
Shymkent     39

Можно ли судить о городе по одному написанию?


На самом деле если мы поймем связь какого-то написания с источником данных, то наверное можно не судить, а проанилизировать, что заставляет отменять заказ в том или ином случае.


Что нужно сделать с городами до анализа (это тема урока 6)?


Нужно провести очистку данных от грязного написания категорий и дублирования строк.
